In [7]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_DIR = '../Data/Raw'
SUB_DIR = '../submissions'
os.makedirs(SUB_DIR, exist_ok=True)

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:

train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Train Dataset Shape: {train.shape}")
print(f"Test Dataset Shape:  {test.shape}")

TARGET = 'addicted_label'
ID_COL = 'id'

base_features = [c for c in train.columns if c not in [TARGET, ID_COL]]

print(f"\nBase Features ({len(base_features)}): {base_features}")
print("\nTarget Class Distribution:")
print(train[TARGET].value_counts(normalize=True).map('{:.2%}'.format))

Train Dataset Shape: (691369, 14)
Test Dataset Shape:  (296302, 13)

Base Features (12): ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']

Target Class Distribution:
addicted_label
1    70.94%
0    29.06%
Name: proportion, dtype: object


In [9]:

def build_features(df_in):
    df = df_in.copy()
    eps = 1e-5 # Prevents division by zero

   
    df['social_media_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + eps)
    df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + eps)
    df['work_study_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + eps)
    df['leisure_hours'] = df['social_media_hours'] + df['gaming_hours']
    df['leisure_ratio'] = df['leisure_hours'] / (df['daily_screen_time_hours'] + eps)

    df['screen_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + eps)
    df['weekend_daily_diff'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
    df['weekend_daily_ratio'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + eps)
    df['active_non_screen_hours'] = 24.0 - (df['daily_screen_time_hours'] + df['sleep_hours'])

    df['notifications_per_screen_hour'] = df['notifications_per_day'] / (df['daily_screen_time_hours'] + eps)
    df['app_opens_per_screen_hour'] = df['app_opens_per_day'] / (df['daily_screen_time_hours'] + eps)
    df['notifications_per_app_open'] = df['notifications_per_day'] / (df['app_opens_per_day'] + eps)

    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

    return df

print("Applying feature engineering pipeline...")
train_fe = build_features(train)
test_fe = build_features(test)


cat_cols = ['gender', 'stress_level', 'academic_work_impact']


train_fe['age_group'] = pd.qcut(train_fe['age'], q=5, labels=False, duplicates='drop').astype(str)
test_fe['age_group'] = pd.qcut(test_fe['age'], q=5, labels=False, duplicates='drop').astype(str)


for cat in ['stress_level', 'age_group']:
    
    group_stats = train_fe.groupby(cat)[['daily_screen_time_hours', 'notifications_per_day']].agg(['mean', 'std'])
    group_stats.columns = [f'{col}_{cat}_{stat}' for col, stat in group_stats.columns]
    
    
    train_fe = train_fe.merge(group_stats, on=cat, how='left')
    test_fe = test_fe.merge(group_stats, on=cat, how='left')

cat_cols_all = cat_cols + ['age_group']
for col in cat_cols_all:
    train_fe[col] = train_fe[col].astype('category')
    test_fe[col] = test_fe[col].astype('category')


feature_cols = [c for c in train_fe.columns if c not in [TARGET, ID_COL]]

print(f"Original Feature Count:  {len(base_features)}")
print(f"Engineered Feature Count: {len(feature_cols)}")

Applying feature engineering pipeline...
Original Feature Count:  12
Engineered Feature Count: 33
